# Incubation Portfolio Optimizer
## A Policy-Aligned Pipeline for Evidence-Based Venture Selection

This notebook implements a reproducible two-stage pipeline for incubator portfolio design
under government-mandated multi-objective constraints:

**Stage 1 — Predictive Modeling**
- Ridge-penalized logistic regression per outcome, penalty selected by LOOCV (log-loss)
- Out-of-sample performance: AUC, Log-Loss, Brier score via LOOCV
- Permutation tests for significance of predictive signal
- Calibration assessment: reliability curves, Brier decomposition, isotonic post-calibration
- Average Marginal Effects (AMEs) with bootstrap 95% CIs

**Stage 2 — Portfolio Optimization**
- Per-venture objective coefficients from AMEs
- Multi-Objective Binary Integer Program (MO-BIP) solved with PuLP/CBC
- ε-constraint method to trace Pareto frontiers under capacity and composition constraints
- Monte Carlo uncertainty propagation (AME + baseline sampling)
- Evidence-weighted objectives (optional: scale by permutation p-value confidence)

**Outcomes** (binary, mapped to government KPI categories):

| Variable | Policy KPI |
|---|---|
| `ip` | Intellectual property / technology transfer |
| `hub` | Regional anchoring / science park affiliation |
| `rev_high` | Revenue / commercialization impact |
| `team_growth` | Employment generation |
| `stage_growth` | Technological stage advancement |

**Predictors**:

| Variable | Type | Description |
|---|---|---|
| `tech_digital` | binary | Tech domain: 1=digital/platform/Industry-4.0, 0=bio/agri |
| `founders` | integer | Number of founding partners |
| `incub_years` | numeric | Incubation duration (years) |
| `stage_in` | ordinal (1–4) | Technology stage at entry: 1=Ideation → 4=Scale-up |
| `team_size_in` | binary | Team size bracket at entry (0=<5, 1=5–10 employees) |
| `stage_adv` | binary | Advanced stage at entry: `stage_in` > 2 |

**Usage**: Set `DATA_PATH` in *Section 1* to your cleaned dataset (`data/venture_cohort.csv`).
Adjust `OUTCOME_SPECS` to match your covariates.
All random operations are seeded for full reproducibility.

---
*Reference*: [Author(s) omitted for review]. Designing Incubation Portfolios Aligned
with Public Policy Objectives. *Technovation* (under review).


## Section 1 — Setup & Configuration

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
# If this cell raises ModuleNotFoundError, install dependencies first:
#   pip install -r requirements.txt
import warnings, json
warnings.simplefilter("ignore", UserWarning)
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import LeaveOneOut, StratifiedKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    from sklearn.calibration import calibration_curve, CalibratedClassifierCV
    from scipy.special import expit
    import pulp
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"Missing package: {{_e}}. Run: pip install -r requirements.txt"
    ) from _e

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print("All imports OK.")

In [ ]:
from pathlib import Path  # ensure Path is available even if Cell 1 was not run


# ── File paths (update DATA_PATH to your cleaned CSV) ────────────────────────
DATA_PATH      = Path("data/venture_cohort.csv")        # output of data_preparation step
DICT_PATH      = Path("data/venture_cohort_dictionary.json")
OUTPUTS_DIR    = Path("outputs_elasticnet")
OUTPUTS_DIR.mkdir(exist_ok=True)
# ── Reproducibility seeds ─────────────────────────────────────────────────────
SEED_PERM  = 42     # permutation tests
SEED_BOOT  = 123    # pairs bootstrap
SEED_MC    = 42     # Monte Carlo
# ── Hyperparameters ───────────────────────────────────────────────────────────
C_GRID        = np.logspace(-3, 3, 13)   # penalty grid
L1_RATIO_GRID = [0.5]  # Strategy 2: fixed l1_ratio=0.5 — tuning two HPs is too noisy for n=19
                         # Change to [0.1,0.3,0.5,0.7,0.9] to re-enable full grid
B_PERM    = 2_000                      # permutation draws
R_BOOT    = 2_000                      # bootstrap resamples
MC_DRAWS  = 2_000                      # Monte Carlo draws
CAPACITY  = 8                          # incubator slot capacity (K)
ALPHAS    = [0.4, 0.6, 0.8, 1.0]      # Expand-vs-Admit gain scaling
EPS_POINTS = 10                             # ε values per secondary (6^4=1296 instances)
EPS_VEC   = np.linspace(0.05, 0.95, 19)  # ε grid for Monte Carlo scenario sweep
print("Configuration loaded.")
print(f"  Data : {DATA_PATH}")
print(f"  B_PERM={B_PERM}, R_BOOT={R_BOOT}, MC_DRAWS={MC_DRAWS}, K={CAPACITY}")


In [ ]:
# ── Outcome specification: ElasticNet with per-outcome curated predictors ──────
#
# Strategy: for hub / rev_high / team_growth, mirror the Ridge curated sets to
# eliminate noise from irrelevant features (n=19 is very sensitive to this).
#
# Key changes vs previous version:
#   - hub:         only incub_years + stage_in + interaction  (Ridge spec)
#   - rev_high:    only incub_years + team_size_in            (Ridge spec, prescreening off)
#   - team_growth: stage_adv + tech_digital as binaries, NO stage_in cat
#                  → restores the Ridge signal WITHOUT the collinearity issue
#                    (stage_adv only clashes with stage_in dummies; removing stage_in fixes it)
#   - ip / stage_growth: unchanged (full set with l1_ratio=0.5)

_BIN_NO_ADV = ["tech_digital", "team_size_in"]
_ALL_NUM    = ["founders", "incub_years"]
_ALL_CAT    = ["stage_in"]

_INTERACT_NUM_CAT = [("incub_years", "stage_in"), ("founders", "stage_in")]
_INTERACT_NUM_BIN = [("founders",    "tech_digital"),
                     ("incub_years", "team_size_in")]

OUTCOME_SPECS = {
    "ip": {
        "bin_cols":         _BIN_NO_ADV,
        "num_cols":         _ALL_NUM,
        "cat_cols":         _ALL_CAT,
        "interact_num_cat": _INTERACT_NUM_CAT,
        "interact_num_bin": _INTERACT_NUM_BIN,
        "interact_all":     False,
        "l1_ratio_grid":    [0.5],
        "prescreening":     True,
    },
    "hub": {
        # Curated to match Ridge: incub_years + stage_in + interaction only
        # (r=0.360 and -0.222; other vars add noise with n=19)
        "bin_cols":         [],
        "num_cols":         ["incub_years"],
        "cat_cols":         ["stage_in"],
        "interact_num_cat": [("incub_years", "stage_in")],
        "interact_num_bin": [],
        "interact_all":     False,
        "l1_ratio_grid":    [0.1],     # near-Ridge: preserves interaction signal
        "prescreening":     False,     # small feature set — screening not needed
    },
    "rev_high": {
        # Curated to match Ridge: incub_years + team_size_in only
        # (r=-0.387 and 0.347; stage_in adds noise here)
        "bin_cols":         ["team_size_in"],
        "num_cols":         ["incub_years"],
        "cat_cols":         [],
        "interact_num_cat": [],
        "interact_num_bin": [("incub_years", "team_size_in")],
        "interact_all":     False,
        "l1_ratio_grid":    [0.1],     # near-Ridge: better LogLoss calibration
        "prescreening":     False,     # 3 features — screening not needed
    },
    "team_growth": {
        # Restore stage_adv + tech_digital (Ridge spec) WITHOUT stage_in cat
        # stage_adv only causes collinearity against stage_in dummies;
        # removing stage_in eliminates the conflict and recovers the threshold signal
        "bin_cols":         ["tech_digital", "stage_adv"],
        "num_cols":         [],
        "cat_cols":         [],
        "interact_num_cat": [],
        "interact_num_bin": [],
        "interact_all":     False,
        "l1_ratio_grid":    [0.1],     # near-Ridge: better LogLoss calibration
        "prescreening":     False,     # 2 features — no screening
    },
    "stage_growth": {
        # Curated to match Ridge: stage_adv + team_size_in as binaries, no stage_in cat
        # (r=0.367 and 0.456; prescreening=True kept only interaction terms → empty AMEs)
        # l1_ratio=0.1 (near-Ridge): l1_ratio=0.5 zerava ambos os coefs via LOOCV C=0.032
        # porque team_size_in tem apenas 1 observação positiva em 19 — L1 forte o suficiente
        # para zerar tudo; near-Ridge mantém coefs não-nulos (LogLoss=0.567 vs 0.587 nulo)
        "bin_cols":         ["stage_adv", "team_size_in"],
        "num_cols":         [],
        "cat_cols":         [],
        "interact_num_cat": [],
        "interact_num_bin": [],
        "interact_all":     False,
        "l1_ratio_grid":    [0.1],   # near-Ridge: evita shrinkage completo com n=1 em team_size_in
        "prescreening":     False,
    },
}

PRIMARY   = "rev_high"
ALL_OBJS  = list(OUTCOME_SPECS.keys())
SECONDARY = [o for o in ALL_OBJS if o != PRIMARY]
print("ElasticNet OUTCOME_SPECS — per-outcome curated predictors:", ALL_OBJS)

## Section 2 — Data Preparation

Load the cleaned dataset produced by `data_preparation.py` (or the companion
data-preparation notebook). The expected schema is documented in
`data/venture_cohort_dictionary.json`.

**Binary outcomes** (0/1):
- `ip` — intellectual property registered (patent/software) during or after graduation
- `hub` — post-graduation affiliation with hub/park/innovation space
- `rev_high` — high-revenue bracket (≥ R$5M–R$10M)
- `team_growth` — team headcount bracket grew after graduation
- `stage_growth` — technology maturity stage advanced after graduation

**Predictors**:
- `tech_digital` (0/1): digital/platform/Industry-4.0 tech domain
- `founders` (int): number of founding partners
- `incub_years` (float): incubation duration in years
- `stage_in` (1–4): technology maturity stage at incubation entry
- `team_size_in` (0/1): entry team size bracket (0=<5, 1=5–10)
- `stage_adv` (0/1): advanced entry stage (stage_in > 2)


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {len(df)} ventures × {df.shape[1]} columns")
print(f"\nOutcome prevalences:")
for col in ALL_OBJS:
    if col in df.columns:
        p = pd.to_numeric(df[col], errors="coerce").mean()
        print(f"  {col:12s}: {p:.3f}  ({int(round(p*len(df)))}/{len(df)})")
df.head()

## Section 3 — Helper FunctionsAll methodological steps are encapsulated here. These functions are calledonce per outcome in Section 4 and are fully reusable for other datasets.

In [ ]:
def build_design_matrix(df, spec):
    """
    Build design matrix supporting interact_num_cat AND interact_num_bin.

    interact_num_cat : [(num, cat), ...] — num_std × each cat dummy
    interact_num_bin : [(num, bin), ...] — num_std × binary predictor
    interact_all     : bool — additionally add bin×bin, bin×cat_dummy (rarely used)
    """
    target           = spec["target"]
    bin_cols         = spec.get("bin_cols", [])
    num_cols         = spec.get("num_cols", [])
    cat_cols         = spec.get("cat_cols", [])
    interact_num_cat = spec.get("interact_num_cat", [])
    interact_num_bin = spec.get("interact_num_bin", [])   # Strategy 1
    interact_all     = spec.get("interact_all", False)

    parts, num_used, bin_used, cat_dummies = [], [], [], []
    num_arrays, bin_arrays, dummy_arrays   = {}, {}, {}

    scaler = StandardScaler()
    if num_cols:
        Xn = df[num_cols].apply(pd.to_numeric, errors="coerce")
        Xn_s = scaler.fit_transform(Xn)
        parts.append(Xn_s)
        num_used = num_cols[:]
        for j, col in enumerate(num_cols):
            num_arrays[col] = Xn_s[:, j]

    if bin_cols:
        Xb = df[bin_cols].apply(pd.to_numeric, errors="coerce").values
        parts.append(Xb)
        bin_used = bin_cols[:]
        for k, col in enumerate(bin_cols):
            bin_arrays[col] = Xb[:, k]

    for col in cat_cols:
        dummies = pd.get_dummies(df[col].astype(str), drop_first=True,
                                 dtype=float, prefix=col)
        parts.append(dummies.values)
        for k, dc in enumerate(dummies.columns):
            cat_dummies.append(dc)
            dummy_arrays[dc] = dummies.values[:, k]

    interact_cols = []
    interact_info = []
    existing = set()

    # num × cat dummies
    for num_col, cat_col in interact_num_cat:
        if num_col not in num_arrays:
            continue
        z = num_arrays[num_col]
        for dc in [c for c in cat_dummies if c.startswith(cat_col + "_")]:
            ic = f"{num_col}:{dc}"
            if ic not in existing:
                parts.append((z * dummy_arrays[dc]).reshape(-1, 1))
                interact_cols.append(ic); existing.add(ic)
                interact_info.append({"interact_col": ic, "num_col": num_col,
                                       "cat_dummy": dc, "itype": "num_cat"})

    # num × binary (Strategy 1: curated num×bin interactions)
    for num_col, bin_col in interact_num_bin:
        if num_col not in num_arrays or bin_col not in bin_arrays:
            continue
        ic = f"{num_col}:{bin_col}"
        if ic not in existing:
            parts.append((num_arrays[num_col] * bin_arrays[bin_col]).reshape(-1, 1))
            interact_cols.append(ic); existing.add(ic)
            interact_info.append({"interact_col": ic, "num_col": num_col,
                                   "bin_col": bin_col, "itype": "num_bin"})

    if interact_all:
        # bin × bin
        bl = list(bin_arrays.keys())
        for i_b, bc1 in enumerate(bl):
            for bc2 in bl[i_b+1:]:
                ic = f"{bc1}:{bc2}"
                if ic not in existing:
                    parts.append((bin_arrays[bc1] * bin_arrays[bc2]).reshape(-1, 1))
                    interact_cols.append(ic); existing.add(ic)
                    interact_info.append({"interact_col": ic, "bin_col1": bc1,
                                           "bin_col2": bc2, "itype": "bin_bin"})
        # bin × cat dummy
        for bc, bv in bin_arrays.items():
            for dc, dv in dummy_arrays.items():
                ic = f"{bc}:{dc}"
                if ic not in existing:
                    parts.append((bv * dv).reshape(-1, 1))
                    interact_cols.append(ic); existing.add(ic)
                    interact_info.append({"interact_col": ic, "bin_col": bc,
                                           "cat_dummy": dc, "itype": "bin_cat"})

    X = np.hstack(parts).astype(float)
    y = pd.to_numeric(df[target], errors="coerce").values.astype(float)
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return X[mask], y[mask], scaler, num_used, bin_used, cat_dummies, interact_cols, interact_info

In [ ]:
def fit_loocv_elasticnet(X, y, C_grid, l1_ratio_grid):
    """
    Select ElasticNet C (and optionally l1_ratio) by LOOCV log-loss.
    Strategy 2: l1_ratio_grid=[0.5] fixes mixing, only tunes C.
    Strategy 3: max_iter=20000, tol=1e-5 for proper SAGA convergence.
    """
    loo = LeaveOneOut()
    grid = []
    for l1 in l1_ratio_grid:
        for C in C_grid:
            clf = LogisticRegression(l1_ratio=l1, C=C, solver="saga",
                                     max_iter=20_000, tol=1e-5,
                                     class_weight="balanced",   # prevent null-model win on imbalanced outcomes
                                     random_state=42)
            y_true, y_pred = [], []
            try:
                for tr, te in loo.split(X):
                    clf.fit(X[tr], y[tr])
                    y_pred.append(clf.predict_proba(X[te])[:, 1][0])
                    y_true.append(y[te][0])
                ll = log_loss(y_true, y_pred, labels=[0, 1])
            except Exception:
                ll = np.inf
            grid.append((l1, C, ll))
    grid_df  = pd.DataFrame(grid, columns=["l1_ratio", "C", "loocv_logloss"])
    best_row = grid_df.loc[grid_df["loocv_logloss"].idxmin()]
    return float(best_row["C"]), float(best_row["l1_ratio"]), grid_df

In [ ]:
def _tune_C_inner(X_tr, y_tr, C_grid_inner, seed, l1_ratio_grid=None):
    """Select C and l1_ratio via StratifiedKFold on n-1 samples (ElasticNet)."""
    if l1_ratio_grid is None:
        l1_ratio_grid = [0.5]
    minority = int(np.bincount(y_tr.astype(int)).min())
    n_splits = max(3, min(5, minority))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    best_C, best_l1, best_ll = C_grid_inner[0], l1_ratio_grid[0], np.inf
    for l1 in l1_ratio_grid:
        for C in C_grid_inner:
            clf_i = LogisticRegression(l1_ratio=l1, C=C, solver="saga", max_iter=20_000, tol=1e-5,
                                       class_weight="balanced", random_state=seed)
            lls = []
            for tr_i, va_i in skf.split(X_tr, y_tr):
                try:
                    clf_i.fit(X_tr[tr_i], y_tr[tr_i])
                    p_va = clf_i.predict_proba(X_tr[va_i])[:, 1]
                    lls.append(log_loss(y_tr[va_i], p_va, labels=[0, 1]))
                except Exception:
                    lls.append(np.inf)
            m = float(np.mean(lls))
            if m < best_ll:
                best_ll, best_C, best_l1 = m, C, l1
    return best_C, best_l1


def compute_loocv_metrics_and_permtest(X, y, best_C, B=2_000, seed=42,
                                        checkpoint_file=None,
                                        best_l1=0.5, l1_ratio_grid=None):
    """Nested-CV LOOCV metrics + permutation test with ElasticNet (saga)."""
    import pickle as _pickle
    if l1_ratio_grid is None:
        l1_ratio_grid = L1_RATIO_GRID
    C_INNER   = np.logspace(-3, 3, 7)
    SAVE_EVERY = 200
    loo = LeaveOneOut()
    rng = np.random.default_rng(seed)

    def _loocv_probs(X_, y_, base_seed):
        yt, yp = [], []
        for fold_idx, (tr, te) in enumerate(loo.split(X_)):
            fold_seed = base_seed + fold_idx
            C_fold, l1_fold = _tune_C_inner(X_[tr], y_[tr], C_INNER,
                                             seed=fold_seed,
                                             l1_ratio_grid=l1_ratio_grid)
            clf_f = LogisticRegression(l1_ratio=l1_fold, C=C_fold, solver="saga", max_iter=20_000, tol=1e-5,
                                       class_weight="balanced", random_state=fold_seed)
            try:
                clf_f.fit(X_[tr], y_[tr])
                yp.append(clf_f.predict_proba(X_[te])[:, 1][0])
            except Exception:
                yp.append(float(y_.mean()))
            yt.append(y_[te][0])
        return np.array(yt), np.array(yp)

    y_true, y_prob = _loocv_probs(X, y, base_seed=seed * 13)
    obs_auc = roc_auc_score(y_true, y_prob)
    obs_ll  = log_loss(y_true, y_prob, labels=[0, 1])
    obs_br  = brier_score_loss(y_true, y_prob)

    start_b, null_auc, null_ll, null_br = 0, [], [], []
    if checkpoint_file is not None and Path(checkpoint_file).exists():
        try:
            with open(checkpoint_file, "rb") as _f:
                _ckpt = _pickle.load(_f)
            null_auc = _ckpt["null_auc"]; null_ll = _ckpt["null_ll"]
            null_br  = _ckpt["null_br"]; start_b  = _ckpt["n_completed"]
            rng.bit_generator.state = _ckpt["rng_state"]
            print(f"  [RESUME] {start_b}/{B} draws loaded")
        except Exception as _e:
            print(f"  [WARN] {_e}; restarting")
            start_b, null_auc, null_ll, null_br = 0, [], [], []

    for b in range(start_b, B):
        y_perm = rng.permutation(y)
        try:
            yt_b, yh_b = _loocv_probs(X, y_perm, base_seed=77 + b)
            if len(np.unique(y_perm)) == 2:
                null_auc.append(roc_auc_score(y_perm, yh_b))
            null_ll.append(log_loss(y_perm, yh_b, labels=[0, 1]))
            null_br.append(brier_score_loss(y_perm, yh_b))
        except Exception:
            pass
        if checkpoint_file is not None and (b + 1) % SAVE_EVERY == 0:
            _ckpt = {"null_auc": null_auc, "null_ll": null_ll, "null_br": null_br,
                     "n_completed": b + 1, "rng_state": rng.bit_generator.state}
            with open(checkpoint_file, "wb") as _f:
                _pickle.dump(_ckpt, _f)
            print(f"  [CHECKPOINT] {b+1}/{B} saved")

    if checkpoint_file is not None and Path(checkpoint_file).exists():
        Path(checkpoint_file).unlink()

    def pval_hi(obs, null):
        null = np.array(null); null = null[np.isfinite(null)]
        return (1 + np.sum(null >= obs)) / (len(null) + 1) if len(null) else np.nan
    def pval_lo(obs, null):
        null = np.array(null); null = null[np.isfinite(null)]
        return (1 + np.sum(null <= obs)) / (len(null) + 1) if len(null) else np.nan

    metrics = {
        "AUC": obs_auc, "p_AUC": pval_hi(obs_auc, null_auc),
        "LogLoss": obs_ll, "p_LL": pval_lo(obs_ll, null_ll),
        "Brier": obs_br,   "p_Brier": pval_lo(obs_br, null_br),
        "y_true": y_true, "y_prob": y_prob,
    }

    p_bar = y_true.mean()
    n_bins = max(3, len(y_true) // 5)
    frac_pos, mean_pred = calibration_curve(y_true, y_prob,
                                             n_bins=n_bins, strategy="quantile")
    n_ab = len(frac_pos)
    counts = np.histogram(y_prob, bins=n_ab)[0]
    reliability = np.average((mean_pred - frac_pos) ** 2, weights=counts + 1e-9)
    resolution  = np.average((frac_pos - p_bar) ** 2,  weights=counts + 1e-9)

    isotonic_probs = None
    is_miscalibrated = reliability > 0.01  # lowered threshold: apply calibration more aggressively
    if is_miscalibrated:
        base_clf = LogisticRegression(l1_ratio=best_l1, C=best_C, solver="saga", max_iter=5000)
        minority = int(np.bincount(y.astype(int)).min())
        cal_clf  = CalibratedClassifierCV(
            base_clf, method="isotonic",
            cv=StratifiedKFold(n_splits=max(2, min(5, minority)),
                               shuffle=True, random_state=42))
        cal_clf.fit(X, y)
        isotonic_probs = cal_clf.predict_proba(X)[:, 1]

    calib = {"reliability": reliability, "resolution": resolution,
             "uncertainty": p_bar * (1 - p_bar), "is_miscalibrated": is_miscalibrated,
             "frac_pos": frac_pos, "mean_pred": mean_pred,
             "isotonic_probs": isotonic_probs}
    return metrics, calib

In [ ]:
def compute_ames_with_bootstrap(clf, X, y, num_cols_used, bin_cols_used, scaler,
                               cat_dummies=None, interact_info=None,
                               R=2_000, seed=123):
    """
    Average Marginal Effects with pairs-bootstrap 95%% CIs.

    Handles four interaction types produced by build_design_matrix(interact_all=True):
      num_cat : num_std × cat_dummy  — derivative of num_col accounts for interaction
      num_bin : num_std × binary     — same
      bin_bin : binary × binary      — discrete change of bin_col1 also flips product
      bin_cat : binary × cat_dummy   — discrete change of binary also flips product
                                       discrete change of cat_dummy also flips product

    For cat_dummies: discrete change AME flips the dummy and any interaction column
    that involves it (num_cat → also flip num×d, bin_cat → also flip bin×d).
    """
    cat_dummies   = cat_dummies  or []
    interact_info  = interact_info or []

    n_num = len(num_cols_used)
    n_bin = len(bin_cols_used)
    n_cat = len(cat_dummies)

    int_col_names = [d["interact_col"] for d in interact_info]

    def _num_idx(col):
        return num_cols_used.index(col) if col in num_cols_used else None
    def _bin_idx(col):
        idx = bin_cols_used.index(col) if col in bin_cols_used else None
        return None if idx is None else n_num + idx
    def _cat_idx(col):
        idx = cat_dummies.index(col) if col in cat_dummies else None
        return None if idx is None else n_num + n_bin + idx
    def _int_idx(name):
        idx = int_col_names.index(name) if name in int_col_names else None
        return None if idx is None else n_num + n_bin + n_cat + idx

    def _ames_once(clf_, X_, scaler_=None):
        mu = clf_.predict_proba(X_)[:, 1]
        b  = clf_.coef_.ravel()
        rows = []

        # ── Continuous: derivative AME accounting for num_cat and num_bin ────────
        for j, col in enumerate(num_cols_used):
            sd = scaler_.scale_[j] if (scaler_ is not None
                                        and scaler_.scale_ is not None) else 1.0
            interact_contrib = np.zeros(len(X_))
            for idict in interact_info:
                if idict.get("num_col") != col:
                    continue
                ic_j = _int_idx(idict["interact_col"])
                if ic_j is None:
                    continue
                itype = idict.get("itype", "num_cat")
                if itype == "num_cat":
                    cat_j = _cat_idx(idict.get("cat_dummy", ""))
                    if cat_j is not None:
                        interact_contrib += b[ic_j] * X_[:, cat_j]
                elif itype == "num_bin":
                    bin_j = _bin_idx(idict.get("bin_col", ""))
                    if bin_j is not None:
                        interact_contrib += b[ic_j] * X_[:, bin_j]
            dmu = mu * (1 - mu) * (b[j] + interact_contrib) / sd
            rows.append((col, "continuous", float(np.mean(dmu))))

        # ── Binary: discrete change + update all interaction columns ──────────────
        for k, col in enumerate(bin_cols_used):
            j = _bin_idx(col)
            X0, X1 = X_.copy(), X_.copy()
            X0[:, j] = 0.0
            X1[:, j] = 1.0
            for idict in interact_info:
                ic_j = _int_idx(idict["interact_col"])
                if ic_j is None:
                    continue
                itype = idict.get("itype", "num_cat")
                if itype == "num_bin" and idict.get("bin_col") == col:
                    num_j = _num_idx(idict.get("num_col", ""))
                    if num_j is not None:
                        X0[:, ic_j] = 0.0
                        X1[:, ic_j] = X_[:, num_j]        # num × 1
                elif itype == "bin_bin":
                    bc1 = idict.get("bin_col1", "")
                    bc2 = idict.get("bin_col2", "")
                    if bc1 == col:
                        other_j = _bin_idx(bc2)
                        if other_j is not None:
                            X0[:, ic_j] = 0.0
                            X1[:, ic_j] = X_[:, other_j]  # 1 × bc2
                    elif bc2 == col:
                        other_j = _bin_idx(bc1)
                        if other_j is not None:
                            X0[:, ic_j] = 0.0
                            X1[:, ic_j] = X_[:, other_j]  # bc1 × 1
                elif itype == "bin_cat" and idict.get("bin_col") == col:
                    cat_j = _cat_idx(idict.get("cat_dummy", ""))
                    if cat_j is not None:
                        X0[:, ic_j] = 0.0
                        X1[:, ic_j] = X_[:, cat_j]        # 1 × d_k
            rows.append((col, "binary",
                         float(np.mean(clf_.predict_proba(X1)[:, 1]
                                      - clf_.predict_proba(X0)[:, 1]))))

        # ── Cat dummies: discrete change + update interaction columns ─────────────
        for col in cat_dummies:
            cat_j = _cat_idx(col)
            X0, X1 = X_.copy(), X_.copy()
            X0[:, cat_j] = 0.0
            X1[:, cat_j] = 1.0
            for idict in interact_info:
                ic_j = _int_idx(idict["interact_col"])
                if ic_j is None:
                    continue
                itype = idict.get("itype", "num_cat")
                if itype == "num_cat" and idict.get("cat_dummy") == col:
                    num_j = _num_idx(idict.get("num_col", ""))
                    if num_j is not None:
                        X0[:, ic_j] = 0.0
                        X1[:, ic_j] = X_[:, num_j]
                elif itype == "bin_cat" and idict.get("cat_dummy") == col:
                    bin_j = _bin_idx(idict.get("bin_col", ""))
                    if bin_j is not None:
                        X0[:, ic_j] = 0.0
                        X1[:, ic_j] = X_[:, bin_j]
            rows.append((col, "binary",
                         float(np.mean(clf_.predict_proba(X1)[:, 1]
                                      - clf_.predict_proba(X0)[:, 1]))))
        return rows

    ame_rows = _ames_once(clf, X, scaler)
    ame_df   = pd.DataFrame(ame_rows, columns=["var", "type", "AME"])

    rng  = np.random.default_rng(seed)
    boot = {r[0]: [] for r in ame_rows}
    for _ in range(R):
        idx = rng.integers(0, len(X), size=len(X))
        Xb, yb = X[idx], y[idx]
        sc_b = StandardScaler().fit(Xb[:, :n_num]) if n_num > 0 else None
        try:
            clf_b = LogisticRegression(
                l1_ratio=getattr(clf, "l1_ratio", 0.5), C=clf.C,
                solver="saga", max_iter=20_000, tol=1e-5, random_state=42,
            )
            clf_b.fit(Xb, yb)
            for var, _, val in _ames_once(clf_b, Xb, sc_b):
                boot[var].append(val)
        except Exception:
            pass

    ci_rows = []
    for var, draws in boot.items():
        if draws:
            ci_rows.append({"var": var,
                            "CI_2.5":  float(np.percentile(draws,  2.5)),
                            "CI_97.5": float(np.percentile(draws, 97.5))})
    if ci_rows:
        ci_df  = pd.DataFrame(ci_rows).set_index("var")
        ame_df = ame_df.set_index("var").join(ci_df, how="left").reset_index()
    else:
        # Guard: no CI data (e.g. all bootstrap folds single-class or no AME vars)
        ame_df["CI_2.5"]  = np.nan
        ame_df["CI_97.5"] = np.nan
    return ame_df

In [ ]:
def _screen_features(X, feature_mask,
                     num_used, bin_used, cat_dummies,
                     interact_cols, interact_info):
    """Filter design matrix and metadata to Ridge-selected columns."""
    n_num = len(num_used); n_bin = len(bin_used); n_cat = len(cat_dummies)
    num_kept  = [col for j, col in enumerate(num_used)
                 if feature_mask[j]]
    bin_kept  = [col for j, col in enumerate(bin_used)
                 if feature_mask[n_num + j]]
    cat_kept  = [col for j, col in enumerate(cat_dummies)
                 if feature_mask[n_num + n_bin + j]]
    int_kept  = [col for j, col in enumerate(interact_cols)
                 if feature_mask[n_num + n_bin + n_cat + j]]
    info_kept = [d for d in interact_info if d["interact_col"] in set(int_kept)]
    return X[:, feature_mask], num_kept, bin_kept, cat_kept, int_kept, info_kept


def run_outcome_pipeline(target_col, df, C_grid=C_GRID, B_perm=B_PERM,
                         R_boot=R_BOOT, seed_perm=SEED_PERM, seed_boot=SEED_BOOT,
                         save_prefix=None, checkpoint_dir=None):
    """
    ElasticNet pipeline with per-outcome fine-tuning:
      - l1_ratio from spec["l1_ratio_grid"]  (hub/rev_high/team_growth → 0.1)
      - Ridge pre-screening from spec["prescreening"]  (hub → disabled)
      - Calibration threshold 0.01 (more aggressive)
    """
    import pickle as _pickle

    spec = dict(OUTCOME_SPECS[target_col])
    spec["target"] = target_col

    # Per-outcome hyperparameter settings
    l1_grid    = spec.get("l1_ratio_grid", L1_RATIO_GRID)
    prescreen  = spec.get("prescreening", True)

    (X_full, y, scaler,
     num_full, bin_full, cat_full,
     int_full, info_full) = build_design_matrix(df, spec)

    print(f"\n{chr(9472)*60}")
    print(f"Outcome: {target_col.upper()}  |  n={len(y)}, p0={y.mean():.3f}")
    print(f"  Full feature set: {X_full.shape[1]} cols "
          f"(num:{num_full} bin:{bin_full} cat:{cat_full} interact:{len(int_full)})")
    print(f"  l1_ratio_grid={l1_grid}  pre-screening={'ON' if prescreen else 'OFF'}")

    # ── Ridge pre-screening (per-outcome flag) ────────────────────────────────
    if prescreen and X_full.shape[1] > 2:
        clf_screen = LogisticRegression(l1_ratio=0, C=1.0, solver="lbfgs",
                                        max_iter=5000, random_state=42)
        clf_screen.fit(X_full, y)
        coef_abs  = np.abs(clf_screen.coef_.ravel())
        threshold = coef_abs.mean() + 0.5 * coef_abs.std()
        feat_mask = coef_abs > threshold
        if feat_mask.sum() < 2:
            feat_mask = np.zeros(len(coef_abs), dtype=bool)
            feat_mask[np.argsort(coef_abs)[-2:]] = True
        X, num_used, bin_used, cat_dummies, interact_cols, interact_info =             _screen_features(X_full, feat_mask,
                             num_full, bin_full, cat_full,
                             int_full, info_full)
        print(f"  After pre-screening: {X.shape[1]}/{X_full.shape[1]} features kept")
        print(f"    num:{num_used}  bin:{bin_used}  cat:{cat_dummies}")
        if interact_cols:
            print(f"    interact:{interact_cols}")
    else:
        X, num_used, bin_used, cat_dummies = X_full, num_full, bin_full, cat_full
        interact_cols, interact_info = int_full, info_full
        print(f"  Pre-screening OFF — using all {X.shape[1]} features")

    # ── ElasticNet LOOCV selection ────────────────────────────────────────────
    best_C, best_l1, grid_df = fit_loocv_elasticnet(X, y, C_grid, l1_grid)
    print(f"  Best C={best_C:.4f}, l1_ratio={best_l1:.1f} "
          f"(LOOCV log-loss={grid_df['loocv_logloss'].min():.4f})")

    # ── Final model ───────────────────────────────────────────────────────────
    clf = LogisticRegression(l1_ratio=best_l1, C=best_C, solver="saga",
                             max_iter=20_000, tol=1e-5, random_state=42)
    clf.fit(X, y)

    # ── LOOCV metrics + permutation ───────────────────────────────────────────
    perm_ckpt = None
    if checkpoint_dir is not None:
        Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
        perm_ckpt = Path(checkpoint_dir) / f"{target_col}_perm.pkl"

    metrics, calib = compute_loocv_metrics_and_permtest(
        X, y, best_C, B=B_perm, seed=seed_perm,
        checkpoint_file=perm_ckpt, best_l1=best_l1,
        l1_ratio_grid=l1_grid,
    )
    print(f"  AUC={metrics['AUC']:.4f} (p={metrics['p_AUC']:.4f}) | "
          f"LL={metrics['LogLoss']:.4f} (p={metrics['p_LL']:.4f}) | "
          f"Brier={metrics['Brier']:.4f} (p={metrics['p_Brier']:.4f})")
    if calib["is_miscalibrated"]:
        print(f"  Miscalibration (reliability={calib['reliability']:.4f}); "
              "isotonic applied.")

    # ── AMEs ─────────────────────────────────────────────────────────────────
    ame_df = compute_ames_with_bootstrap(
        clf, X, y, num_used, bin_used, scaler,
        cat_dummies=cat_dummies, interact_info=interact_info,
        R=R_boot, seed=seed_boot,
    )

    if save_prefix:
        pfx = OUTPUTS_DIR / save_prefix
        ame_df.to_csv(f"{pfx}_ames.csv", index=False)
        pd.DataFrame([{k: v for k, v in metrics.items()
                       if k not in ("y_true","y_prob")}
                     ]).to_csv(f"{pfx}_loocv_perm.csv", index=False)

    return dict(metrics=metrics, calib=calib, ame_df=ame_df,
                clf=clf, X=X, y=y, scaler=scaler,
                num_used=num_used, bin_used=bin_used,
                cat_dummies=cat_dummies,
                interact_cols=interact_cols, interact_info=interact_info,
                spec=spec)

print("ElasticNet helper functions defined.")

## Section 4 — Stage 1: Predictive ModelingRun the full pipeline for each outcome. Results are stored in `RESULTS` dict.

In [ ]:
import pickle as _pickle

# Checkpoint directories
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Outcomes to process : {ALL_OBJS}")
print(f"Permutation draws   : {B_PERM}  (saved every 200)")
print()

RESULTS = {}

for outcome in ALL_OBJS:
    outcome_ckpt = CHECKPOINT_DIR / f"{outcome}_result.pkl"

    # ── Load from outcome-level checkpoint if available ───────────────────────
    if outcome_ckpt.exists():
        try:
            with open(outcome_ckpt, "rb") as _f:
                RESULTS[outcome] = _pickle.load(_f)
            m = RESULTS[outcome]["metrics"]
            print(f"\n{chr(9472)*55}")
            print(f"[CHECKPOINT] {outcome.upper()} loaded from disk — skipping recomputation")
            print(f"  AUC={m['AUC']:.4f} (p={m['p_AUC']:.4f}) | "
                  f"LL={m['LogLoss']:.4f} (p={m['p_LL']:.4f}) | "
                  f"Brier={m['Brier']:.4f} (p={m['p_Brier']:.4f})")
            print(f"  AMEs: {RESULTS[outcome]['ame_df']['var'].tolist()}")
            continue
        except Exception as _e:
            print(f"[WARN] Could not load checkpoint for {outcome} ({_e}); recomputing")

    # ── Run full pipeline (permutations auto-checkpointed every 200 draws) ────
    RESULTS[outcome] = run_outcome_pipeline(
        target_col     = outcome,
        df             = df,
        save_prefix    = outcome,
        checkpoint_dir = CHECKPOINT_DIR,
    )

    # ── Save outcome-level checkpoint ─────────────────────────────────────────
    with open(outcome_ckpt, "wb") as _f:
        _pickle.dump(RESULTS[outcome], _f)
    print(f"  [CHECKPOINT] {outcome}_result.pkl saved")

print("\n\nAll outcomes processed.")

In [ ]:
# ── Table: LOOCV metrics and permutation p-values (Table 2 in paper) ─────────
rows = []
for outcome, res in RESULTS.items():
    m = res["metrics"]
    rows.append({"Outcome": outcome,
                 "AUC": m["AUC"],  "p_AUC": m["p_AUC"],
                 "LogLoss": m["LogLoss"], "p_LL": m["p_LL"],
                 "Brier": m["Brier"],   "p_Brier": m["p_Brier"]})

pred_table = pd.DataFrame(rows).set_index("Outcome").round(4)
pred_table.to_csv(OUTPUTS_DIR / "table_loocv_metrics.csv")
print("LOOCV metrics table:")
print(pred_table)

In [ ]:
# ── Table: AMEs with bootstrap CIs (Table 3 in paper) ────────────────────────
ame_tables = {}
for outcome, res in RESULTS.items():
    ame_df = res["ame_df"].copy()
    ame_df["Outcome"] = outcome
    ame_tables[outcome] = ame_df

all_ames = pd.concat(ame_tables.values(), ignore_index=True)
all_ames.to_csv(OUTPUTS_DIR / "table_ames.csv", index=False)
print("AME table (all outcomes):")
print(all_ames.to_string(index=False))

In [ ]:
# ── Calibration reliability curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, len(ALL_OBJS), figsize=(4*len(ALL_OBJS), 4), sharey=True)
for ax, outcome in zip(axes, ALL_OBJS):
    calib = RESULTS[outcome]["calib"]
    ax.plot(calib["mean_pred"], calib["frac_pos"], "s-", label="LOOCV probs")
    if calib["isotonic_probs"] is not None:
        y_true = RESULTS[outcome]["metrics"]["y_true"]
        frac_c, mean_c = calibration_curve(y_true, calib["isotonic_probs"],
                                           n_bins=5, strategy="quantile")
        ax.plot(mean_c, frac_c, "^--", label="Isotonic")
    ax.plot([0,1],[0,1],"k--", alpha=0.4, label="Perfect")
    ax.set_title(outcome); ax.set_xlabel("Mean predicted"); ax.legend(fontsize=7)
axes[0].set_ylabel("Fraction positives")
fig.suptitle("Reliability (Calibration) Curves — LOOCV predictions", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "fig_calibration_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Calibration curves saved.")

## Section 5 — Per-Venture Objective CoefficientsFor each venture *i* and outcome *j*:$$g_{ij} = p_{0j} + \sum_k \text{AME}_{jk} \cdot x_{ik}$$where $p_{0j}$ is the sample prevalence and AME medians are used as point estimates.These coefficients serve as the objective weights $u_{ij}$ in the MILP.

In [ ]:
def build_objective_coeffs(df, ame_df, target_col, scaler=None, num_used=None, bin_used=None):
    """
    Compute per-venture objective coefficient g_i = p0 + sum_k AME_k * x_ik.

    For one-hot dummy columns (e.g. "stage_in_2") that are not in df directly,
    the dummy is reconstructed from the base categorical column.

    Returns a Series indexed by df.index with g_i values in [0, 1] after min-max norm.
    """
    y_vec = pd.to_numeric(df[target_col], errors="coerce")
    p0    = float(y_vec.mean())

    ame_map = dict(zip(ame_df["var"], ame_df["AME"]))
    g = np.full(len(df), p0, dtype=float)

    for var, ame_val in ame_map.items():
        if var in df.columns:
            # Direct column (binary or numeric predictor)
            x_col = pd.to_numeric(df[var], errors="coerce").fillna(0).values
            g += ame_val * x_col
        else:
            # Try to reconstruct as a one-hot dummy: "base_col_level"
            reconstructed = False
            for base_col in df.columns:
                if var.startswith(base_col + "_"):
                    level = var[len(base_col) + 1:]
                    dummy = (df[base_col].astype(str) == level).astype(float).values
                    g += ame_val * dummy
                    reconstructed = True
                    break
            if not reconstructed:
                pass   # silently skip interaction-derived columns

    return pd.Series(g, index=df.index, name=target_col)

df_coeffs = {"startup_id": np.arange(1, len(df)+1)}
for outcome, res in RESULTS.items():
    g = build_objective_coeffs(df, res["ame_df"], outcome,
                               scaler=res["scaler"], num_used=res["num_used"],
                               bin_used=res["bin_used"])
    df_coeffs[outcome] = g.values

COEFFS = pd.DataFrame(df_coeffs)

# Min-max normalize to [0,1] per objective
NORM = COEFFS.copy()
for col in ALL_OBJS:
    lo, hi = COEFFS[col].min(), COEFFS[col].max()
    NORM[col + "_01"] = (COEFFS[col] - lo) / (hi - lo) if hi > lo else 0.0

NORM.to_csv(OUTPUTS_DIR / "objectives_normalized.csv", index=False)
print("Per-venture coefficients computed and normalized.")
print(NORM[[c + "_01" for c in ALL_OBJS]].describe().round(3))

## Section 6 — Stage 2: ε-Constraint Portfolio OptimizationSolves the capacity-constrained MO-BIP:$$\max_{x \in \{0,1\}^n} \sum_i u_{i,\text{primary}} x_i$$$$\text{s.t.} \quad \sum_i x_i = K, \quad \sum_i u_{ij} x_i \geq \varepsilon_j \; \forall j \neq \text{primary}$$**Evidence-weighted variant** (optional): scale each $u_{ij}$ by a confidencefactor derived from permutation p-values before optimization:$$\tilde{u}_{ij} = u_{ij} \cdot w_j, \quad w_j = 1 - p_j^{\text{perm}}$$Set `USE_EVIDENCE_WEIGHTS=True` to activate.

In [ ]:
# ── Evidence-weighting (optional) ─────────────────────────────────────────────
USE_EVIDENCE_WEIGHTS = False   # set True for the evidence-weighted variant

obj_cols_norm = [c + "_01" for c in ALL_OBJS]

if USE_EVIDENCE_WEIGHTS:
    # Weight each objective by (1 - permutation p-value for AUC)
    ew = {}
    for col, outcome in zip(obj_cols_norm, ALL_OBJS):
        p_auc = RESULTS[outcome]["metrics"]["p_AUC"]
        ew[col] = max(0.0, 1.0 - p_auc)
    print("Evidence weights:", {k: round(v, 3) for k, v in ew.items()})
else:
    ew = {c: 1.0 for c in obj_cols_norm}
    print("Evidence weights: uniform (1.0 per objective)")

In [ ]:
import itertools as _itertools

def _topk_upper(DF_norm, col, K):
    """Sum of the K largest normalized values for col — absolute upper bound on ε floor."""
    vals = DF_norm[col].sort_values(ascending=False).values
    return float(np.sum(vals[:K])) if len(vals) >= K else float(np.sum(vals))


def _eps_grid_for(DF_norm, col, K, n_pts):
    """
    Build an n_pts ε grid for `col` that covers the FULL range [0, topK] but is
    denser at low values (where most feasible solutions cluster) and still reaches
    the top (where specialised, diversified portfolios appear).

    Strategy: concatenate a fine linear grid over [0, 0.5*ub] with a coarser one
    over [0.5*ub, ub], then deduplicate and sort.
    """
    ub   = _topk_upper(DF_norm, col, K)
    n_lo = int(np.ceil(n_pts * 0.7))          # 70 % of points in lower half
    n_hi = max(2, n_pts - n_lo)               # remaining in upper half
    lo   = np.linspace(0.0,       0.5 * ub, n_lo)
    hi   = np.linspace(0.5 * ub,       ub,  n_hi)
    pts  = np.unique(np.concatenate([lo, hi]))
    return pts.tolist()


def _dominance_mask(points):
    """True if point is non-dominated (maximisation in all dimensions)."""
    n = points.shape[0]
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]:
            continue
        pi = points[i]
        for j in range(n):
            if i == j or not keep[j]:
                continue
            pj = points[j]
            if np.all(pj >= pi) and np.any(pj > pi):
                keep[i] = False
                break
    return keep


def _hypervolume_2d(points, ref=(0.0, 0.0)):
    """
    2D hypervolume (maximisation) with reference point ref.
    Filters to non-dominated set, sorts by x, integrates swept area.
    """
    if points.size == 0:
        return 0.0
    pts = points[_dominance_mask(points)]
    if pts.shape[0] == 0:
        return 0.0
    pts = pts[np.argsort(pts[:, 0])]
    hv, x_prev, y_best = 0.0, ref[0], ref[1]
    for x, y in pts:
        hv    += max(0.0, x - x_prev) * max(0.0, y - ref[1])
        x_prev = x
        y_best = max(y_best, y)
    return float(max(0.0, hv))


def solve_eps_constraint(DF_norm, primary_col, secondary_cols,
                          K, eps_grid=None, eps_points=6,
                          obj_weights=None, verbose=False):
    """
    Sweep the ε-constraint MILP.

    Grid strategy (matching original script):
      For each secondary, build an independent ε-grid from 0 to its top-K upper
      bound with eps_points values.  The full Cartesian product (eps_points ^ n_sec
      instances) is solved and deduplicated on solution signature (objective sums).

    For backward-compatible single-grid usage, pass eps_grid (1-D array) and
    eps_points is ignored; the same ε is applied uniformly to all secondaries.

    Parameters
    ----------
    eps_grid   : 1-D array or None.  If None, per-secondary grids are used.
    eps_points : int — points per secondary grid (default 6 → 6^4=1296 instances).

    Returns
    -------
    frontier : DataFrame  [eps_<sec>, status, gap, solve_time, obj_value,
                            selected, *secondary_sums]
               Deduplicated on solution signature; Coverage_eps = feas / total.
    """
    if obj_weights is None:
        obj_weights = {c: 1.0 for c in [primary_col] + secondary_cols}

    ids = DF_norm["startup_id"].astype(int).tolist()

    # Build per-secondary ε grids
    uniform_mode = eps_grid is not None
    if uniform_mode:
        # Uniform mode: same ε for every secondary
        combos_raw = list(eps_grid)
        combos = [dict(zip(secondary_cols, [eps] * len(secondary_cols)))
                  for eps in combos_raw]
    else:
        sec_grids = {}
        for sec in secondary_cols:
            # Dense-then-coarse grid over full [0, topK] range
            sec_grids[sec] = _eps_grid_for(DF_norm, sec, K, eps_points)
        combos = [dict(zip(secondary_cols, vals))
                  for vals in _itertools.product(*[sec_grids[s] for s in secondary_cols])]

    rows = []
    for eps_floors in combos:
        prob = pulp.LpProblem("eps_constraint", pulp.LpMaximize)
        x    = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in ids}

        prob += pulp.lpSum(
            obj_weights.get(primary_col, 1.0) *
            float(DF_norm.loc[DF_norm["startup_id"]==i, primary_col].values[0]) * x[i]
            for i in ids)
        prob += (pulp.lpSum(x[i] for i in ids) == K), "capacity"
        for sec, floor_val in eps_floors.items():
            prob += (pulp.lpSum(
                obj_weights.get(sec, 1.0) *
                float(DF_norm.loc[DF_norm["startup_id"]==i, sec].values[0]) * x[i]
                for i in ids) >= floor_val), f"floor_{sec}"

        prob.solve(pulp.PULP_CBC_CMD(msg=int(verbose)))

        status     = pulp.LpStatus[prob.status]
        solve_time = float(prob.solutionTime)
        gap        = 0.0 if status == "Optimal" else float("nan")
        obj_val    = pulp.value(prob.objective) if status == "Optimal" else float("nan")
        selected   = ([i for i in ids if pulp.value(x[i]) and pulp.value(x[i]) > 0.5]
                      if status == "Optimal" else [])
        sec_sums   = {}
        for sec in secondary_cols:
            if status == "Optimal":
                sec_sums[sec] = sum(
                    float(DF_norm.loc[DF_norm["startup_id"]==i, sec].values[0])
                    for i in selected)
            else:
                sec_sums[sec] = float("nan")

        # In uniform mode add single "eps" column for MC loop compatibility
        if uniform_mode:
            common_eps = eps_floors[secondary_cols[0]]
            eps_cols = {"eps": common_eps,
                        **{f"eps_{s}": v for s, v in eps_floors.items()}}
        else:
            eps_cols = {f"eps_{s}": v for s, v in eps_floors.items()}
        row = {**eps_cols,
               "status": status, "gap": gap, "solve_time": solve_time,
               "obj_value": obj_val, "selected": selected, **sec_sums}
        rows.append(row)

    df = pd.DataFrame(rows)

    # Deduplicate on solution signature (same portfolio → same objective sums)
    sig_cols = [sec for sec in secondary_cols] + ["obj_value"]
    df_dedup = df.drop_duplicates(subset=sig_cols, keep="first").reset_index(drop=True)
    n_raw, n_dedup = len(df), len(df_dedup)
    feas_raw = (df["status"] == "Optimal").sum()
    print(f"  Instances: {n_raw} | Feasible: {feas_raw} ({feas_raw/n_raw:.1%}) | "
          f"Unique after dedup: {n_dedup}")
    return df_dedup

print("MILP solver function defined.")

In [ ]:
# ── Run ε-constraint sweep ────────────────────────────────────────────────────
PRIMARY_COL   = PRIMARY + "_01"
SECONDARY_COLS= [c + "_01" for c in SECONDARY]

frontier = solve_eps_constraint(
    DF_norm        = NORM,
    primary_col    = PRIMARY_COL,
    secondary_cols = SECONDARY_COLS,
    K              = CAPACITY,
    eps_points     = EPS_POINTS,   # per-secondary grid: EPS_POINTS^4 instances
    obj_weights    = ew,
)

feasible  = frontier[frontier["status"]=="Optimal"]
n_total   = len(frontier)
n_feasible= len(feasible)
print(f"Instances: {n_total} | Feasible: {n_feasible} ({n_feasible/n_total:.1%})")
print(f"Optimality gap: always 0.00 for feasible instances (CBC exact solver)")
frontier.to_csv(OUTPUTS_DIR / "frontier.csv", index=False)
frontier.head(8)

In [ ]:
# ── Pareto front scatter (Figure: front_scatter) ─────────────────────────────
feasible = frontier[frontier["status"] == "Optimal"].copy()
if len(feasible) > 0:
    y = feasible["obj_value"].to_numpy(dtype=float)
    x = feasible[SECONDARY_COLS].mean(axis=1).to_numpy(dtype=float)

    def _mm(a):
        lo, hi = np.min(a), np.max(a)
        return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

    x_n, y_n = _mm(x), _mm(y)

    plt.figure(figsize=(5.6, 4.2))
    plt.scatter(x_n, y_n, s=28, alpha=0.7, edgecolors="none")
    plt.xlabel("Mean of secondaries (min–max normalized on frontier)")
    plt.ylabel("Primary — rev_high (min–max normalized on frontier)")
    plt.title("Pareto Front — ε-constraint sweep")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "front_scatter.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"front_scatter.png saved  ({len(feasible)} feasible points)")

## Section 7 — Stage 3: Monte Carlo Uncertainty PropagationPropagates uncertainty in AMEs (sampled from Normal distributions implied bybootstrap CIs) and baseline rates (Beta priors) across `MC_DRAWS` draws.For each draw, the ε-constraint problem is solved for a policy scenario comparingtwo candidate options (*Expand* an incumbent vs. *Admit* a waitlist entrant)across a grid of α (Expand gain scaling) and ε (secondary floor) values.**Outputs**: selection probabilities, feasibility surfaces, Pareto front bands,switching thresholds, and robust rankings.

In [ ]:
# ── AME uncertainty sampling ──────────────────────────────────────────────────
def sample_ames(RESULTS, rng):
    """
    Sample one draw of AMEs per outcome from Normal(median, se_approx).
    se_approx = (CI_97.5 - CI_2.5) / (2 * 1.96)
    """
    sampled = {}
    for outcome, res in RESULTS.items():
        ame_df = res["ame_df"].copy()
        vals = []
        for _, row in ame_df.iterrows():
            lo, hi = row.get("CI_2.5", row["AME"]), row.get("CI_97.5", row["AME"])
            se = (hi - lo) / (2 * 1.96) if (hi - lo) > 0 else 1e-6
            vals.append(rng.normal(loc=row["AME"], scale=se))
        sampled[outcome] = pd.DataFrame({"var": ame_df["var"].values, "AME": vals})
    return sampled

# ── Baseline rate sampling ─────────────────────────────────────────────────────
def sample_baselines(df, ALL_OBJS, rng):
    """Sample baseline prevalences p0 from Beta(a, b) priors."""
    p0s = {}
    for col in ALL_OBJS:
        y_  = pd.to_numeric(df[col], errors="coerce").dropna()
        a, b_ = float(y_.sum()) + 0.5, float((1-y_).sum()) + 0.5
        p0s[col] = float(rng.beta(a, b_))
    return p0s

print("Monte Carlo sampling functions defined.")

In [ ]:
# ── Synthetic cohort for Expand vs Admit scenario ────────────────────────────
# Variable mapping (pipeline names → original names):
#   founders      = owners     | tech_digital = tecnologia | stage_in  = sin
#   incub_years   = tempo_incub| team_size_in = team_entry | stage_adv = shigh

N_INCUB = 7   # incumbent ventures  (labels I1 … I7)
N_WAIT  = 3   # waitlisted ventures (labels W1 … W3)

def _clamp(arr, allowed):
    """Snap each element to the nearest value in the allowed set."""
    allowed = np.array(sorted(allowed), dtype=float)
    arr = np.asarray(arr, dtype=float)
    return allowed[np.abs(arr[:, None] - allowed[None, :]).argmin(axis=1)].astype(int)


def generate_synthetic_cohort(df_obs, n_incub=N_INCUB, n_wait=N_WAIT,
                               seed=SEED_MC):
    """
    Sample a synthetic decision pool calibrated to the observed venture
    distribution.

    Incumbents (group="incubated", labels I1…I_n_incub):
      attributes drawn from observed empirical distribution with slight
      positive jitter (longer incubation, slightly larger teams).

    Waitlisted (group="waitlist", labels W1…W_n_wait):
      same distribution but shifted toward earlier stages and smaller teams
      (younger, less developed ventures on the waitlist).

    Returns a DataFrame saved as outputs/synthetic_cohort.csv.
    """
    rng = np.random.default_rng(seed)

    obs = df_obs.copy()
    # Observed distributions (for sampling)
    founders_vals  = obs["founders"].dropna().astype(int).values
    tech_vals      = obs["tech_digital"].dropna().astype(int).values
    stage_vals     = obs["stage_in"].dropna().astype(int).values
    incub_vals     = obs["incub_years"].dropna().astype(int).values
    team_vals      = obs["team_size_in"].dropna().astype(int).values

    def sample_cat(vals, n):
        uv, cnt = np.unique(vals, return_counts=True)
        return rng.choice(uv, size=n, p=cnt/cnt.sum())

    def sample_int_jitter(vals, n, jitter=0.08, minval=1):
        base = rng.choice(vals.astype(float), size=n, replace=True)
        noise = rng.lognormal(mean=0.0, sigma=jitter, size=n)
        return np.maximum(minval, np.round(base * noise)).astype(int)

    # Incumbents — calibrated to observed distribution
    incub = pd.DataFrame({
        "founders":     sample_int_jitter(founders_vals, n_incub, jitter=0.08),
        "tech_digital": sample_cat(tech_vals, n_incub),
        "stage_in":     _clamp(sample_cat(stage_vals, n_incub), {1, 2, 3, 4}),
        "incub_years":  sample_int_jitter(incub_vals, n_incub, jitter=0.05),
        "team_size_in": sample_cat(team_vals, n_incub),
    })

    # Waitlisted — slightly earlier stage, less incubation time
    wait = pd.DataFrame({
        "founders":     np.maximum(1, sample_int_jitter(founders_vals, n_wait, jitter=0.10) - 1),
        "tech_digital": sample_cat(tech_vals, n_wait),
        "stage_in":     _clamp(sample_cat(stage_vals, n_wait).astype(float) - 0.3,
                                {1, 2, 3, 4}),
        "incub_years":  np.maximum(1, sample_int_jitter(incub_vals, n_wait, jitter=0.10) - 1),
        "team_size_in": _clamp(sample_cat(team_vals, n_wait).astype(float) - 0.3,
                                {0, 1}),
    })

    incub.insert(0, "startup_id", [f"I{k+1}" for k in range(n_incub)])
    incub.insert(1, "group",      "incubated")
    wait.insert( 0, "startup_id", [f"W{k+1}" for k in range(n_wait)])
    wait.insert( 1, "group",      "waitlist")

    cohort = pd.concat([incub, wait], ignore_index=True)

    # Derived features needed by AME-based prediction
    cohort["stage_adv"] = (cohort["stage_in"] > 2).astype(int)

    cohort.to_csv(OUTPUTS_DIR / "synthetic_cohort.csv", index=False)
    return cohort


def choose_by_eps(pool, eps, secondary_cols, primary_col):
    """
    Greedy ε-selection: among pool members with all secondaries ≥ eps,
    return the one with the highest primary objective.  If none feasible,
    return the best primary without floor constraint (infeasible flag).

    pool : DataFrame with columns [startup_id, group, option, *obj_cols]
    """
    feas = pool.copy()
    for sec in secondary_cols:
        feas = feas[feas[sec] + 1e-9 >= eps]
    if feas.empty:
        best = pool.sort_values(primary_col, ascending=False).iloc[0]
        return {"feasible": False, "chosen": best["startup_id"],
                "option": best["option"],
                **{c: float(best[c]) for c in [primary_col] + secondary_cols}}
    best = feas.sort_values(primary_col, ascending=False).iloc[0]
    return {"feasible": True, "chosen": best["startup_id"],
            "option": best["option"],
            **{c: float(best[c]) for c in [primary_col] + secondary_cols}}


# ── Generate synthetic cohort ─────────────────────────────────────────────────
cohort_df = generate_synthetic_cohort(df, N_INCUB, N_WAIT, seed=SEED_MC)

print(f"Synthetic cohort: {N_INCUB} incumbents (I1–I{N_INCUB}) + "
      f"{N_WAIT} waitlisted (W1–W{N_WAIT})")
print(cohort_df[["startup_id", "group", "stage_in", "stage_adv",
                  "incub_years", "tech_digital", "founders",
                  "team_size_in"]].to_string(index=False))

In [ ]:
# ── Monte Carlo: Expand vs Admit uncertainty propagation ─────────────────────
#
# Each draw:
#   1. Sample AME for each outcome from Normal(bootstrap_median, CI/1.96/2)
#   2. Sample p0 for each outcome from Beta(obs_successes+1, obs_failures+1)
#   3. Compute g_ij = p0_j + sum_k AME_jk * x_ik  for every synthetic venture
#   4. Build pool: Expand = incumbent g_ij × alpha; Admit = waitlist g_ij as-is
#   5. Greedy ε-selection: pick best primary among feasible (all sec ≥ eps)

MC_PRIMARY_COL    = PRIMARY
MC_SECONDARY_COLS = SECONDARY

rng_mc = np.random.default_rng(SEED_MC)
log_rows = []

# Observed prevalences for re-centering
p0_obs = {outcome: float(df[outcome].mean()) for outcome in ALL_OBJS}


def _g_ij_from_ame(venture_row, ame_df_draw, p0):
    """
    Compute g_ij = p0 + sum_k AME_k * x_ik  for a single venture.

    venture_row : pd.Series with predictor values (no outcome column needed)
    ame_df_draw : DataFrame with columns [var, AME]
    p0          : float — sampled baseline probability for this outcome
    """
    g = p0
    for _, ame_row in ame_df_draw.iterrows():
        var = ame_row["var"]
        ame_val = float(ame_row["AME"])
        if var in venture_row.index:
            x_val = pd.to_numeric(venture_row[var], errors="coerce")
            if pd.notna(x_val):
                g += ame_val * float(x_val)
        else:
            # Reconstruct one-hot dummy (e.g. "stage_in_2" from "stage_in")
            for base_col in venture_row.index:
                if var.startswith(base_col + "_"):
                    level = var[len(base_col) + 1:]
                    dummy = float(str(venture_row[base_col]) == level)
                    g += ame_val * dummy
                    break
    return float(np.clip(g, 0.0, 1.0))


for draw in range(MC_DRAWS):

    # Step 1-2: sample AMEs and baselines
    sampled_ames = sample_ames(RESULTS, rng_mc)
    p0s          = sample_baselines(df, ALL_OBJS, rng_mc)

    # Step 3: compute g_ij for every synthetic venture
    preds = cohort_df[["startup_id", "group"]].copy()
    for outcome in ALL_OBJS:
        ame_draw = sampled_ames[outcome]
        g_vals = [
            _g_ij_from_ame(row, ame_draw, p0s[outcome])
            for _, row in cohort_df.iterrows()
        ]
        preds[outcome] = g_vals

    # Step 4-5: sweep alpha and eps
    for alpha in ALPHAS:
        pool_parts = []
        for _, row in preds.iterrows():
            r = row.to_dict()
            if r["group"] == "incubated":
                r["option"] = "Expand"
                for outcome in ALL_OBJS:
                    r[outcome] = float(np.clip(alpha * r[outcome], 0.0, 1.0))
            else:
                r["option"] = "Admit"
            pool_parts.append(r)
        pool = pd.DataFrame(pool_parts)

        for eps in EPS_VEC:
            res = choose_by_eps(
                pool, float(eps),
                secondary_cols=MC_SECONDARY_COLS,
                primary_col=MC_PRIMARY_COL,
            )
            log_rows.append({
                "draw": draw, "alpha": float(alpha), "eps": float(eps),
                **res,
            })

    if (draw + 1) % 200 == 0:
        print(f"  MC draw {draw+1}/{MC_DRAWS}")

log_df = pd.DataFrame(log_rows)
log_df.to_csv(OUTPUTS_DIR / "mc_selection_log.csv", index=False)
print(f"Monte Carlo complete — {len(log_df):,} rows logged.")
print(f"Option breakdown:\n{log_df['option'].value_counts().to_string()}")

In [ ]:
# ── MC Summary: P(Feasible) and P(Expand) heatmaps ───────────────────────────

agg = (log_df.groupby(["alpha", "eps"])
       .apply(lambda g: pd.Series({
           "P_feasible": g["feasible"].mean(),
           "P_expand":   (g.loc[g["feasible"], "option"] == "Expand").mean()
                         if g["feasible"].any() else 0.0,
           "rev_high":   g.loc[g["feasible"], MC_PRIMARY_COL].mean()
                         if g["feasible"].any() else float("nan"),
       }))
       .reset_index())

agg.to_csv(OUTPUTS_DIR / "mc_heatmap_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, title in zip(
        axes,
        ["P_feasible", "P_expand"],
        ["(a) P(Feasible)", "(b) P(Expand | Feasible)"]):
    pivot = agg.pivot(index="alpha", columns="eps", values=col)
    im = ax.imshow(pivot.values, aspect="auto", origin="lower",
                   vmin=0, vmax=1)
    # String labels on both axes
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:.2f}" for v in pivot.columns],
                       rotation=90, fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{a:g}" for a in pivot.index])
    ax.set_xlabel("ε (secondary floor)")
    ax.set_ylabel("α (expansion credit)")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="Probability")

plt.suptitle("Monte Carlo — Expand vs Admit Policy Scenario")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "mc_heat_expand.png", dpi=200, bbox_inches="tight")
plt.show()
print("mc_heat_expand.png saved.")
print(agg.groupby("alpha")[["P_feasible", "P_expand"]].mean().round(3).to_string())

In [ ]:
# ── MC front bands figure — mc_front_bands.png ───────────────────────────────
# For each (alpha, eps) group across all draws: compute 5th/50th/95th
# quantiles of (mean_secondaries, primary) and plot as a scatter coloured by ε.

sec_cols = MC_SECONDARY_COLS   # ip, hub, team_growth, stage_growth

band_rows = []
for (alpha, eps), g in log_df.groupby(["alpha", "eps"]):
    for var, vals in [("sec_avg", g[sec_cols].mean(axis=1)),
                      (MC_PRIMARY_COL, g[MC_PRIMARY_COL])]:
        arr = vals.dropna().to_numpy()
        if len(arr) == 0:
            continue
        q = np.quantile(arr, [0.05, 0.5, 0.95])
        band_rows.append({"alpha": alpha, "eps": eps, "var": var,
                          "q05": q[0], "median": q[1], "q95": q[2]})

band = pd.DataFrame(band_rows)
band.to_csv(OUTPUTS_DIR / "mc_front_bands.csv", index=False)

# ── Figure: 2×2 subplots, one per alpha ──────────────────────────────────────
fig, axs = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
fig.subplots_adjust(right=0.78, hspace=0.35, wspace=0.25)

cbar_axes = [fig.add_axes([0.80, 0.55, 0.018, 0.35]),   # Q95
             fig.add_axes([0.84, 0.55, 0.018, 0.35]),   # Median
             fig.add_axes([0.88, 0.55, 0.018, 0.35])]   # Q05

for ax, (panel, alpha) in zip(axs.flat, enumerate(ALPHAS)):
    sub = band[band["alpha"] == alpha]
    eps_vals = sub[sub["var"] == "sec_avg"]["eps"].values

    x_med = sub[sub["var"] == "sec_avg"]["median"].values
    x_q05 = sub[sub["var"] == "sec_avg"]["q05"].values
    x_q95 = sub[sub["var"] == "sec_avg"]["q95"].values
    y_med = sub[sub["var"] == MC_PRIMARY_COL]["median"].values
    y_q05 = sub[sub["var"] == MC_PRIMARY_COL]["q05"].values
    y_q95 = sub[sub["var"] == MC_PRIMARY_COL]["q95"].values

    sc1 = ax.scatter(x_q95, y_q95, c=eps_vals, cmap="Greens", s=20, alpha=0.8)
    sc2 = ax.scatter(x_med, y_med, c=eps_vals, cmap="Blues",  s=20, alpha=0.8)
    sc3 = ax.scatter(x_q05, y_q05, c=eps_vals, cmap="Reds",   s=20, alpha=0.8)

    ax.set_title(f"({'abcd'[panel]}) α = {alpha:g}")
    if panel in (0, 2):
        ax.set_ylabel(f"Primary — {MC_PRIMARY_COL} (normalized)")
    if panel in (2, 3):
        ax.set_xlabel("Mean of secondaries (normalized)")

fig.colorbar(sc1, cax=cbar_axes[0], label="Q95")
fig.colorbar(sc2, cax=cbar_axes[1], label="Median")
fig.colorbar(sc3, cax=cbar_axes[2], label="Q05")

plt.suptitle("Monte Carlo Pareto Front Bands by α", y=1.01)
plt.savefig(OUTPUTS_DIR / "mc_front_bands.png", dpi=200, bbox_inches="tight")
plt.show()
print("mc_front_bands.png saved.")

In [ ]:
# ── MC Summary: Switching thresholds ─────────────────────────────────────────
# For each draw × alpha, find the lowest ε where the dominant option switches
# from Admit to Expand (or vice-versa).  Only feasible rows are considered.

threshold_rows = []
feasible_log = log_df[log_df["feasible"]].copy()

for alpha, g_alpha in feasible_log.groupby("alpha"):
    switches = []
    for draw_id, dg in g_alpha.groupby("draw"):
        dg = dg.sort_values("eps").reset_index(drop=True)
        switch_eps = float("nan")
        for k in range(1, len(dg)):
            if dg.loc[k, "option"] != dg.loc[k - 1, "option"]:
                switch_eps = float(dg.loc[k, "eps"])
                break
        if not np.isnan(switch_eps):
            switches.append(switch_eps)

    coverage = len(switches) / MC_DRAWS
    if switches:
        threshold_rows.append({
            "alpha":             alpha,
            "eps_switch_median": np.median(switches),
            "eps_switch_p25":    np.percentile(switches, 25),
            "eps_switch_p75":    np.percentile(switches, 75),
            "coverage":          round(coverage, 4),
        })
    else:
        threshold_rows.append({
            "alpha":             alpha,
            "eps_switch_median": float("nan"),
            "eps_switch_p25":    float("nan"),
            "eps_switch_p75":    float("nan"),
            "coverage":          0.0,
        })

thresh_df = pd.DataFrame(threshold_rows)
thresh_df.to_csv(OUTPUTS_DIR / "mc_thresholds.csv", index=False)
print("Switching thresholds (Table mc-thresholds):")
print(thresh_df.round(3).to_string(index=False))

In [ ]:
# ── MC robustness: selection frequency by venture ────────────────────────────
# For each venture in the synthetic cohort, count how often it was chosen
# across all draws × alpha × eps combinations, and compute mean objectives
# conditional on selection.  Matches original script's mc_robustness().

obj_cols = ALL_OBJS   # [rev_high, ip, hub, team_growth, stage_growth]

ch = log_df[log_df["chosen"].notna()].copy()
total_rows = len(log_df)

# Selection counts and mean objectives when chosen
sel_counts = (ch.groupby("chosen")["chosen"]
                .count()
                .rename("n_selected"))
obj_means  = ch.groupby("chosen")[obj_cols].mean()

# Join group label (incubated / waitlist) from synthetic cohort
grp = (cohort_df[["startup_id", "group"]]
       .drop_duplicates()
       .rename(columns={"startup_id": "chosen"}))

robustness = (
    sel_counts.to_frame()
    .join(obj_means, how="left")
    .reset_index()
    .merge(grp, on="chosen", how="left")
)
robustness["select_freq"] = robustness["n_selected"] / float(total_rows)
robustness = robustness.sort_values("select_freq", ascending=False).reset_index(drop=True)

robustness.to_csv(OUTPUTS_DIR / "mc_robustness_top_candidates.csv", index=False)

print("mc_robustness_top_candidates.csv saved.")
print(robustness[["chosen", "group", "select_freq"] + obj_cols].round(3).to_string(index=False))

## Section 8 — MILP Diagnostics

In [ ]:
# ── MILP diagnostics and front quality — Table 6 and Table 7 ─────────────────

feasible = frontier[frontier["status"] == "Optimal"].copy()
feas_mask = frontier["status"] == "Optimal"

n_total    = len(frontier)
n_feasible = int(feas_mask.sum())
feas_rate  = n_feasible / n_total if n_total > 0 else float("nan")
opt_rate   = 1.0   # CBC certifies optimality on every feasible instance

median_gap   = float(frontier.loc[feas_mask, "gap"].median())   if feas_mask.any() else float("nan")
median_stime = float(frontier.loc[feas_mask, "solve_time"].median()) if feas_mask.any() else float("nan")
p95_stime    = float(frontier.loc[feas_mask, "solve_time"].quantile(0.95)) if feas_mask.any() else float("nan")

# Secondary mean column for plots / HV
sec_mean_col = "sec_mean"
frontier[sec_mean_col] = frontier[SECONDARY_COLS].mean(axis=1)
if len(feasible) > 0:
    feasible[sec_mean_col] = feasible[SECONDARY_COLS].mean(axis=1)

# ── Front quality ─────────────────────────────────────────────────────────────
if len(feasible) > 0:
    y_raw = feasible["obj_value"].to_numpy(dtype=float)
    x_raw = feasible[sec_mean_col].to_numpy(dtype=float)

    def _mm(a):
        lo, hi = np.min(a), np.max(a)
        return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

    x_n, y_n = _mm(x_raw), _mm(y_raw)
    pts = np.c_[x_n, y_n]

    # True 2D hypervolume (with reference at origin, after normalization)
    hv_2d        = _hypervolume_2d(pts, ref=(0.0, 0.0))
    # Spread = std of normalized primary (matches original front_quality)
    spread_primary = float(np.std(y_n))
    # ND share via dominance mask
    nd_share       = float(np.mean(_dominance_mask(pts)))
    cardinality    = int(n_feasible)
    coverage_eps   = feas_rate
else:
    hv_2d = spread_primary = nd_share = coverage_eps = float("nan")
    cardinality = 0

# ── Print ─────────────────────────────────────────────────────────────────────
print("=" * 58)
print("TABLE 6 — MILP diagnostics (CBC)")
print(f"  Total instances (unique) : {n_total}")
print(f"  P(Feasible)              : {feas_rate:.3f}")
print(f"  P(Optimal | Feasible)    : {opt_rate:.3f}")
print(f"  Median relative gap      : {median_gap:.4f}")
print(f"  Median solve time (s)    : {median_stime:.4f}")
print(f"  95th-pct solve time (s)  : {p95_stime:.4f}")
print()
print("TABLE 7 — Pareto front quality")
print(f"  HV_2D                    : {hv_2d:.3f}")
print(f"  Spread(Primary) [std]    : {spread_primary:.3f}")
print(f"  Cardinality              : {cardinality}")
print(f"  ND Share                 : {nd_share:.3f}")
print(f"  Coverage_ε               : {coverage_eps:.3f}")
print("=" * 58)

# ── Save CSVs ─────────────────────────────────────────────────────────────────
pd.DataFrame([{
    "n_total": n_total, "n_feasible": n_feasible,
    "P_feasible": round(feas_rate, 4),
    "P_optimal_given_feasible": round(opt_rate, 4),
    "median_gap": round(median_gap, 4),
    "median_solve_time_s": round(median_stime, 4),
    "p95_solve_time_s": round(p95_stime, 4),
}]).to_csv(OUTPUTS_DIR / "table6_milp_diagnostics.csv", index=False)

pd.DataFrame([{
    "HV_2D": round(hv_2d, 4),
    "spread_primary_std": round(spread_primary, 4),
    "cardinality": cardinality,
    "nd_share": round(nd_share, 4),
    "coverage_eps": round(coverage_eps, 4),
}]).to_csv(OUTPUTS_DIR / "table7_front_quality.csv", index=False)

frontier.loc[feas_mask, [c for c in frontier.columns
                          if c.startswith("eps_")] + ["solve_time", "gap"]].to_csv(
    OUTPUTS_DIR / "milp_solve_times.csv", index=False)

print("Saved: table6_milp_diagnostics.csv | table7_front_quality.csv | milp_solve_times.csv")

In [ ]:
# ── ECDF of solve times — Figure 1 in paper ──────────────────────────────────
feas_mask = frontier["status"] == "Optimal"

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

# (a) Solve time ECDF
times = frontier.loc[feas_mask, "solve_time"].to_numpy(dtype=float)
if len(times) > 0:
    t_sorted = np.sort(times)
    F_t = np.arange(1, len(t_sorted) + 1) / len(t_sorted)
    axes[0].plot(t_sorted, F_t, lw=2)
    axes[0].set_xlabel("Solve time (s)")
    axes[0].set_ylabel("ECDF")
    axes[0].set_title("Empirical CDF of solve times")
    axes[0].grid(True, alpha=0.25)

# (b) Gap ECDF (all feasible — gap = 0.0, but shown for completeness)
gaps = frontier.loc[feas_mask, "gap"].dropna().to_numpy(dtype=float)
if len(gaps) > 0:
    g_sorted = np.sort(gaps)
    F_g = np.arange(1, len(g_sorted) + 1) / len(g_sorted)
    axes[1].plot(g_sorted, F_g, lw=2)
    axes[1].set_xlabel("CBC MIP gap")
    axes[1].set_ylabel("ECDF")
    axes[1].set_title("Empirical CDF of MIP gaps")
    axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "ecdf_time.png", dpi=200, bbox_inches="tight")
plt.show()
print("ecdf_time.png saved")

## Section 9 — Export Summary

In [ ]:
print("Files written to", OUTPUTS_DIR)
for f in sorted(OUTPUTS_DIR.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:45s} {size:>8,} bytes")